In [10]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

print("BERT ke liye zaroori libraries import ho gayi hain!")
# Check karein ki GPU available hai ya nahi (Colab par runtime GPU zaroor select karein)
print("GPU Available:", torch.cuda.is_available())

BERT ke liye zaroori libraries import ho gayi hain!
GPU Available: False
BERT ke liye zaroori libraries import ho gayi hain!
GPU Available: False


In [2]:
!pip install --upgrade datasets huggingface_hub

In [3]:
print("Dataset load ho raha hai...")
# Dataset humne fancyzhx wala use kiya hai
dataset = load_dataset("fancyzhx/ag_news")

# FIXED: range keyword hata kar direct range() function pass kiya hai
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print("Ek sample data:", train_dataset[0])

Dataset load ho raha hai...


Train samples: 2000, Test samples: 500
Ek sample data: {'text': 'Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.', 'label': 0}


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    # Text ko tokenize aur fix length (128) mein convert karein
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Data ko tokenize kiya ja raha hai...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Tokenization mukammal!")

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Data ko tokenize kiya ja raha hai...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization mukammal!


In [5]:
# BERT Model load karein classification ke liye
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# Evaluation Metric Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

print("BERT Model aur Metrics tayar hain.")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT Model aur Metrics tayar hain.


In [6]:
training_args = TrainingArguments(
    output_dir="./results",          # Model checkpoints save karne ki jagah
    num_train_epochs=1,              # Sirf 1 epoch taake jaldi complete ho
    per_device_train_batch_size=8,   # Batch size
    per_device_eval_batch_size=8,
    eval_strategy="epoch",     # Har epoch ke baad test data par check karein
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True
)

# Hugging Face Trainer API setup karein
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Trainer successfully configure ho gaya.")

Trainer successfully configure ho gaya.


In [7]:
print("Fine-tuning shuru ho rahi hai... (Isme thoda time lag sakta hai)")
trainer.train()
print("Training mukammal ho gayi!")

Fine-tuning shuru ho rahi hai... (Isme thoda time lag sakta hai)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.346770,0.390182,0.878000,0.877741


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training mukammal ho gayi!


In [8]:
eval_results = trainer.evaluate()
print("\n=== BERT MODEL EVALUATION ===")
print(f"Final Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Final F1-Score: {eval_results['eval_f1']:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.346770,0.390182,1,0.878000,0.877741



=== BERT MODEL EVALUATION ===
Final Accuracy: 0.8780
Final F1-Score: 0.8777


In [9]:
model.save_pretrained("./my_bert_news_model")
tokenizer.save_pretrained("./my_bert_news_model")

print("Mubarak ho! Aapka BERT model './my_bert_news_model' folder mein save ho gaya hai.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mubarak ho! Aapka BERT model './my_bert_news_model' folder mein save ho gaya hai.
